In [ ]:
import wrds
import pandas as pd
from influxdb_client_3 import InfluxDBClient3, Point, WriteOptions, write_client_options

In [ ]:
wrds_db = wrds.Connection()

In [ ]:
def success(self, data: str):
    print(f"Successfully wrote batch: data")

def errcolumnsf, data: str, exception):
    print(f"Failed writing batch: config: {self}, data: {data} due: {exception}")

def retry(self, data: str, exception):
    print(f"Failed retry writing batch: config: {self}, data: {data} retry: {exception}")

# Configure options for batch writing.
write_options = WriteOptions(batch_size=10_000,
                                    flush_interval=10_000,
                                    jitter_interval=2_000,
                                    retry_interval=5_000,
                                    max_retries=5,
                                    max_retry_delay=30_000,
                                    exponential_base=2)

# Create an options dict that sets callbacks and WriteOptions.
wco = write_client_options(success_callback=success,
                          error_callback=error,
                          retry_callback=retry,
                          write_options=write_options)

client = InfluxDBClient3(token=f"apiv3_hoBjjFB7ZYYtYy33a_Ui5VTjsWG848623y8Mlf_a_2gVU86YB8gwVjJVYRJXaTUWnS4desa23-y7DcXa4qxmIg", 
                         host=f"http://localhost:8181", database=f"jkp_factors", write_client_options=wco)

In [ ]:
schema = wrds_db.get_table(library="contrib", table="global_factor", obs=0)
schema.columns

In [ ]:
characteristics = ["ret_exc"]

sql_query= f"""SELECT id, eom, gvkey, permno, size_grp, {' ,'.join(characteristics)}
FROM contrib.global_factor
WHERE common=1 and exch_main=1 and primary_sec=1 and obs_main=1 and
excntry='USA'
"""
usa_data = wrds_db.raw_sql(sql_query)

In [ ]:
import numpy as np

short_data = usa_data.copy().set_index("eom")
short_data["size_grp"] = short_data["size_grp"].fillna("none")
short_data["permno"] = short_data["permno"].astype("string").fillna("none")
short_data["gvkey"] = short_data["gvkey"].fillna("none")

short_data

In [ ]:
client.write(short_data, 
             data_frame_measurement_name="characteristics", 
             data_frame_tag_columns=["size_grp", "gvkey", "permno"])

In [ ]:
factor_returns = pd.read_csv("factor-returns.csv")
factor_returns = factor_returns.set_index("date")
factor_returns

In [ ]:
client.write(factor_returns, 
             data_frame_measurement_name="factor_returns", 
             data_frame_tag_columns=["location", "name", "freq","weighting"])

In [ ]:
wrds_db.list_tables(library="comp")

In [ ]:
comp_comp = wrds_db.get_table(library="comp", table="company")
comp_keys = comp_comp.gvkey.unique()

In [ ]:
wrds_db.list_tables(library="crsp")

In [ ]:
comp_crsp_linker = wrds_db.get_table(library="crsp", table="ccmxpf_lnkhist")

In [ ]:
link_keys = comp_crsp_linker.gvkey.unique()

In [ ]:
comp_comp[comp_comp.gvkey=="011271"]

In [ ]:
a = set(comp_keys)
b = set(link_keys)
a-b